#**Task3:Music Generation with AI**

In [15]:
!pip install music21 tensorflow numpy

 #Import Python Libraries

In [16]:
import numpy as np
import glob
from music21 import converter, instrument, note, chord, stream
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

 #Upload MIDI Dataset

In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("soumikrakshit/classical-music-midi")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'classical-music-midi' dataset.
Path to dataset files: /kaggle/input/classical-music-midi


 #Extract Notes from MIDI

In [10]:
notes = []

files_list = glob.glob(str(path) + "/**/*.mid", recursive=True)

for file in files_list:
    midi = converter.parse(file)

    parts = instrument.partitionByInstrument(midi)

    if parts:
        notes_to_parse = parts.parts[0].recurse()
    else:
        notes_to_parse = midi.flat.notes

    for element in notes_to_parse:
        if isinstance(element, note.Note):
            notes.append(str(element.pitch))
        elif isinstance(element, chord.Chord):
            notes.append('.'.join(str(n) for n in element.normalOrder))

print("Total notes:", len(notes))

Total notes: 11362


 #Prepare Sequences

In [11]:
sequence_length = 50

pitchnames = sorted(set(notes))
note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

X = []
y = []

for i in range(0, len(notes) - sequence_length):
    seq_in = notes[i:i+sequence_length]
    seq_out = notes[i+sequence_length]

    X.append([note_to_int[n] for n in seq_in])
    y.append(note_to_int[seq_out])

n_patterns = len(X)

X = np.reshape(X, (n_patterns, sequence_length, 1))
X = X / float(len(pitchnames))
y = to_categorical(y)

 #Build LSTM Model

In [12]:
model = Sequential()
model.add(LSTM(256, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.3))
model.add(LSTM(256))
model.add(Dense(128, activation='relu'))
model.add(Dense(len(pitchnames), activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')

 #Train Model

In [18]:
model.fit(X, y, epochs=20, batch_size=64)

Epoch 1/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 99s 525ms/step - loss: 4.7648
Epoch 2/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 89s 503ms/step - loss: 4.5120
Epoch 3/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 87s 490ms/step - loss: 4.3756
Epoch 4/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 87s 489ms/step - loss: 4.3308
Epoch 5/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 143s 495ms/step - loss: 4.2696
Epoch 6/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 87s 490ms/step - loss: 4.2377
Epoch 7/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 85s 480ms/step - loss: 4.2127
Epoch 8/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 86s 486ms/step - loss: 4.1641
Epoch 9/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 86s 489ms/step - loss: 4.1401
Epoch 10/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 97s 549ms/step - loss: 4.1250
Epoch 11/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 104s 586ms/step - loss: 4.0704
Epoch 12/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 133s 538ms/step - loss: 4.0252
Epoch 13/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 88s 495ms/step - loss: 3.9750
Epoch 14/20
177/177 ━━━━━━━━━━━━━━━━━━━━ 87s 494ms/step - loss: 3.9328
Epoch 15/20


 #Generate Music

In [14]:
int_to_note = dict((number, note) for number, note in enumerate(pitchnames))

start = np.random.randint(0, len(X)-1)
pattern = X[start]

generated_notes = []

for i in range(100):
    prediction = model.predict(pattern.reshape(1, sequence_length, 1), verbose=0)
    index = np.argmax(prediction)
    result = int_to_note[index]

    generated_notes.append(result)

    pattern = np.append(pattern, [[index/float(len(pitchnames))]], axis=0)
    pattern = pattern[1:]

 #Convert to MIDI

In [19]:
output_notes = []

for pattern in generated_notes:
    if '.' in pattern:
        notes_in_chord = pattern.split('.')
        chord_notes = []

        for n in notes_in_chord:
            # Ensure each part of the chord is treated as a pitch class, assuming they are 0-11
            # If they were full MIDI numbers, int(n) is fine, but for normalOrder, it's pitch class.
            try:
                numeric_n = int(n)
                # music21.note.Note can take pitchClass, which assigns a default octave
                new_note = note.Note(pitchClass=numeric_n)
            except ValueError:
                # Fallback if it's not a simple number, though unlikely for chord parts
                new_note = note.Note(n)
            chord_notes.append(new_note)

        new_chord = chord.Chord(chord_notes)
        output_notes.append(new_chord)

    else:
        # For single notes, 'pattern' can be like 'C4', 'E-3', or a single pitch class number like '0', '9'.
        try:
            # Attempt to convert the pattern string to an integer.
            numeric_val = int(pattern)
            # If it's a pitch class (0-11), create a note using pitchClass.
            # music21 will assign a default octave for these (usually octave 4).
            if 0 <= numeric_val <= 11:
                new_note = note.Note(pitchClass=numeric_val)
            # If it's a number corresponding to a MIDI value (12-127), treat it as a MIDI number.
            elif 12 <= numeric_val <= 127:
                new_note = note.Note(midi=numeric_val)
            else:
                # For other integer values outside common pitch class/MIDI ranges,
                # fall back to treating it as a pitch name string. This might still fail if invalid.
                new_note = note.Note(pattern)
        except ValueError:
            # If conversion to int fails, it means 'pattern' is not a simple integer string.
            # It must be a note name string like 'C4' or 'E-3'.
            new_note = note.Note(pattern)
        output_notes.append(new_note)

midi_stream = stream.Stream(output_notes)
midi_stream.write('midi', fp='generated_music.mid')

'generated_music.mid'

 #Download Output

In [20]:
from google.colab import files
files.download('generated_music.mid')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#**THANK YOU**
Done by Biswaini chhatoi